# 06a · Revision runs on **A100** (10 runs, ≈ 2.5 h)
LOIO baseline, seed 1, SegFormer-B2 (R-S1). Specification: `specifications/02_spec_revision_runs.md`.
**Use an A100 for this notebook** (same GPU as the seed-0 SegFormer-B2 LOIO runs). Resumable: if Colab disconnects, *Run all* again.

In [ ]:
# --- Setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/gwfss_stem_seg'
DATA_ROOT = '/content/gwfss'                      # local disk: fast I/O, rebuilt each session
SPLIT_FILE = f'{PROJECT}/splits/splits.json'      # frozen once, shared by all runs
RESULTS = f'{PROJECT}/results'                    # checkpoints, logs, metrics (on Drive)

import sys, os
sys.path.insert(0, f'{PROJECT}/src')
!pip -q install segmentation-models-pytorch==0.5.0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from gwfss_stem import data
manifest = data.prepare_dataset(DATA_ROOT)   # downloads GWFSS_v1.0 from Hugging Face (~600 MB) if not present
print(len(manifest), 'images')

In [ ]:
from gwfss_stem import plan, engine
import pandas as pd
runs = [r for r in plan.revision_plan() if r['sq']=='R-S1' and r['backbone']=='segformer_b2']
def status(r):
    d = os.path.join(RESULTS, engine.run_name({**engine.DEFAULTS, **r}))
    if os.path.exists(f'{d}/final.pt'): return 'done'
    if os.path.exists(f'{d}/RUNNING'): return 'running/locked'
    if os.path.exists(f'{d}/last.pt'): return 'partial'
    return 'pending'
table = pd.DataFrame([{**r, 'status': status(r)} for r in runs])
display(table.status.value_counts()); display(table)

In [ ]:
for r in runs:
    cfg = {k: v for k, v in r.items() if k != 'sq'}
    run_dir = engine.train_run(cfg, DATA_ROOT, SPLIT_FILE, RESULTS)
    if run_dir:
        engine.evaluate_run(run_dir, DATA_ROOT, SPLIT_FILE, subset='test', tag='ss')
        if cfg['split'].startswith('loio'):
            engine.offset_sweep(run_dir, DATA_ROOT, SPLIT_FILE, subset='val')
            engine.offset_sweep(run_dir, DATA_ROOT, SPLIT_FILE, subset='test')
print('done')